[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/Multimodal-Deep-Learning/blob/main/04_Finetuning_LowCompute/01_lora_from_scratch/01_lora_from_scratch.ipynb)

# 01. LoRA from Scratch — Build It, Understand It

**LoRA (Low-Rank Adaptation)** is THE technique for finetuning large models with limited compute.

**This notebook covers:**
- LoRA math — why low-rank works (with visual proof)
- Build a LoRA layer from scratch in PyTorch
- Apply LoRA to a transformer — see parameter savings
- Compare: Full finetuning vs LoRA vs Frozen
- Rank selection guide (r=4, 8, 16, 32)

---

In [ ]:
# ============================================================
#  Colab Setup (run this cell first if on Google Colab)
# ============================================================
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = "https://github.com/Gaurav14cs17/Multimodal-Deep-Learning.git"
    REPO_DIR = "/content/Multimodal-Deep-Learning"

    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
        !pip install -q -r {REPO_DIR}/requirements.txt

    os.chdir(f"{REPO_DIR}/04_Finetuning_LowCompute/01_lora_from_scratch")
    os.makedirs(f"{REPO_DIR}/assets", exist_ok=True)
    print(f"Colab ready — working in {os.getcwd()}")
else:
    os.makedirs("../assets", exist_ok=True)

In [ ]:
import sys
sys.path.append('../..')

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from utils.visualization import *
from utils.helpers import count_parameters

set_style()

## 1. The Key Insight: Weight Updates Are Low-Rank

**Research finding:** When you finetune a large model, the weight changes $\Delta W$ have low intrinsic rank.  
This means: instead of updating all $d \times d$ parameters, we can approximate $\Delta W \approx B \times A$ where:
- $A$ is $r \times d$ (down-projection)
- $B$ is $d \times r$ (up-projection)
- $r \ll d$ (rank is much smaller than dimension)

**Savings:** $d^2$ → $2dr$ parameters. For $d=768, r=8$: 589,824 → 12,288 (48× fewer!)

### SVD and the Eckart-Young Theorem

Any matrix $W \in \mathbb{R}^{m \times n}$ can be decomposed as:

$$W = U \Sigma V^T$$

where $U$ and $V$ are orthogonal and $\Sigma$ is diagonal with singular values $\sigma_1 \geq \sigma_2 \geq \cdots \geq \sigma_r$.

The **Eckart-Young theorem** states: the best rank-$r$ approximation (minimizing Frobenius norm) is:

$$W_r = U_r \Sigma_r V_r^T$$

**Approximation error:**

$$\|W - W_r\|_F^2 = \sum_{i=r+1}^{R} \sigma_i^2$$

This is the theoretical foundation of LoRA — finetuning updates live in a low-dimensional subspace spanned by the top singular vectors.

### Intrinsic Dimensionality

Research (Aghajanyan et al., 2021) showed that pretrained models have very low **intrinsic dimensionality** — you can optimize in a random $d$-dimensional subspace and still match full finetuning.

For **RoBERTa-base**: intrinsic dim $\approx 200$ out of **125M parameters**!

This means **>99.99%** of parameters are redundant for task adaptation. LoRA exploits this by learning only in a small rank-$r$ subspace instead of the full parameter space.

In [ ]:
# Visual proof: random matrices have low effective rank

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Why Low-Rank Works: Singular Value Analysis', fontsize=16, fontweight='bold')

for ax, d, title in zip(axes, [64, 256, 768], 
                         ['Small (d=64)', 'Medium (d=256)', 'Large (d=768)']):
    # Simulate weight change during finetuning
    # The change tends to be low-rank in practice
    W_pretrained = torch.randn(d, d) * 0.02
    rank_true = 8  # simulate that true update is low rank
    delta_W = torch.randn(d, rank_true) @ torch.randn(rank_true, d) * 0.01
    delta_W += torch.randn(d, d) * 0.001  # small noise
    
    U, S, V = torch.svd(delta_W)
    S_normalized = S / S.sum()
    cumulative = torch.cumsum(S_normalized, dim=0)
    
    ax.bar(range(min(30, d)), S_normalized[:30].numpy(), color='#3498DB', alpha=0.7, label='Singular values')
    ax2 = ax.twinx()
    ax2.plot(range(min(30, d)), cumulative[:30].numpy(), color='#E74C3C', linewidth=2, label='Cumulative')
    ax2.axhline(y=0.95, color='#2ECC71', linestyle='--', label='95% energy')
    ax2.set_ylabel('Cumulative Energy')
    ax2.set_ylim(0, 1.1)
    
    # Find rank needed for 95%
    rank_95 = (cumulative >= 0.95).nonzero()[0].item() + 1
    ax.axvline(x=rank_95, color='#2ECC71', linestyle='--')
    ax.set_title(f'{title}\n95% energy at rank {rank_95} (of {d})')
    ax.set_xlabel('Rank')
    ax.set_ylabel('Singular Value')

plt.tight_layout()
plt.savefig('../assets/low_rank_proof.png', dpi=150, bbox_inches='tight')
plt.show()
print("Most of the 'energy' in weight updates is captured by just a few dimensions!")
print("This is why LoRA works — we only need to learn the important directions.")

In [ ]:
# Draw the LoRA architecture diagram
fig = draw_lora_diagram()
plt.savefig('../assets/lora_architecture.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Build LoRA from Scratch

In [ ]:
class LoRALayer(nn.Module):
    """LoRA adapter layer — adds low-rank update to a frozen linear layer."""
    def __init__(self, original_layer, rank=8, alpha=16):
        super().__init__()
        self.original_layer = original_layer
        in_features = original_layer.in_features
        out_features = original_layer.out_features
        
        # Freeze original weights
        for param in self.original_layer.parameters():
            param.requires_grad = False
        
        # LoRA matrices
        self.lora_A = nn.Parameter(torch.randn(in_features, rank) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(rank, out_features))
        
        # Scaling factor
        self.scaling = alpha / rank
    
    def forward(self, x):
        # Original output (frozen)
        original_output = self.original_layer(x)
        
        # LoRA output (trainable)
        lora_output = (x @ self.lora_A) @ self.lora_B * self.scaling
        
        # h = Wx + BAx * scaling
        return original_output + lora_output
    
    def merge(self):
        """Merge LoRA weights into original layer (for inference)."""
        self.original_layer.weight.data += (
            self.lora_B.T @ self.lora_A.T * self.scaling
        )
        return self.original_layer


# Demo
original = nn.Linear(768, 768)
lora_layer = LoRALayer(original, rank=8, alpha=16)

x = torch.randn(2, 768)
output = lora_layer(x)

orig_params = 768 * 768  # 589,824
lora_params = 768 * 8 + 8 * 768  # 12,288

print(f"Original params: {orig_params:,}")
print(f"LoRA params:     {lora_params:,} ({lora_params/orig_params*100:.1f}%)")
print(f"Savings:         {(1 - lora_params/orig_params)*100:.1f}% fewer parameters!")
print(f"\nOutput shape: {output.shape}")

### LoRA Initialization & Scaling

- **A** is initialized from $\mathcal{N}(0, \sigma^2)$ — Gaussian initialization
- **B** is initialized to **zeros** — so LoRA starts as identity ($\Delta W = BA = 0$)
- **Scaling factor:** $\alpha/r$ where $\alpha$ is a hyperparameter. The output is:

$$h = Wx + \frac{\alpha}{r}BAx$$

- **Why $\alpha/r$?** When you change rank $r$, the output magnitude stays roughly constant. Common: $\alpha = 2r$.
- **Merging at inference:** $W' = W + \frac{\alpha}{r}BA$ — zero additional latency!

In [ ]:
# Visualize parameter savings at different ranks

dims = [128, 256, 512, 768, 1024]
ranks = [2, 4, 8, 16, 32]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Absolute params
ax = axes[0]
for rank in ranks:
    params = [2 * d * rank for d in dims]
    ax.plot(dims, [p/1000 for p in params], 'o-', linewidth=2, label=f'rank={rank}')
full_params = [d*d for d in dims]
ax.plot(dims, [p/1000 for p in full_params], 's--', linewidth=2, 
        color='black', label='Full (d×d)', alpha=0.5)
ax.set_xlabel('Hidden Dimension (d)')
ax.set_ylabel('Parameters (K)')
ax.set_title('LoRA vs Full Parameters', fontsize=14, fontweight='bold')
ax.legend()
ax.set_yscale('log')

# Percentage savings
ax = axes[1]
for d in [128, 256, 768]:
    savings = [(1 - 2*d*r / (d*d)) * 100 for r in ranks]
    ax.plot(ranks, savings, 'o-', linewidth=2, label=f'd={d}')
ax.set_xlabel('LoRA Rank (r)')
ax.set_ylabel('Parameter Savings (%)')
ax.set_title('Parameter Savings by Rank', fontsize=14, fontweight='bold')
ax.legend()
ax.set_ylim(80, 100)

plt.tight_layout()
plt.savefig('../assets/lora_savings.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Apply LoRA to a Transformer

### Which Layers to Apply LoRA

| Target | Typical Choice | Why |
|--------|---------------|-----|
| Q, V projections | Default (LoRA paper) | Q captures queries, V captures values |
| Q, K, V, O | Better quality | Full attention coverage |
| Q, K, V, O + FFN | Best quality | Most parameters, diminishing returns |
| FFN only | Sometimes | If attention is already good |

**Rule of thumb:** More layers = better quality but more memory. Start with Q, V. CLIP applies LoRA to all attention projections.

In [ ]:
def add_lora_to_model(model, rank=8, alpha=16, target_modules=['q_proj', 'v_proj']):
    """Add LoRA to specific linear layers in a model."""
    lora_layers = []
    
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            # In practice, apply to Q and V projections in attention
            should_apply = any(t in name for t in target_modules) if target_modules else True
            if should_apply or not target_modules:
                parent_name = '.'.join(name.split('.')[:-1])
                child_name = name.split('.')[-1]
                parent = model
                for part in parent_name.split('.'):
                    if part:
                        parent = getattr(parent, part)
                
                lora = LoRALayer(module, rank=rank, alpha=alpha)
                setattr(parent, child_name, lora)
                lora_layers.append(name)
    
    return lora_layers


# Create a transformer model
class SmallTransformer(nn.Module):
    def __init__(self, dim=256, n_layers=4, n_heads=4, n_classes=10):
        super().__init__()
        self.embed = nn.Linear(dim, dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=dim, nhead=n_heads, dim_feedforward=dim*4, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.head = nn.Linear(dim, n_classes)
    
    def forward(self, x):
        x = self.embed(x)
        x = self.transformer(x)
        return self.head(x[:, 0])


# Compare: Full vs LoRA
model_full = SmallTransformer(dim=256, n_layers=4)
model_lora = SmallTransformer(dim=256, n_layers=4)

# Freeze all parameters, then add LoRA
for param in model_lora.parameters():
    param.requires_grad = False

# Add LoRA to all linear layers
lora_layers = add_lora_to_model(model_lora, rank=8, target_modules=[])
print(f"LoRA applied to {len(lora_layers)} layers\n")

# Compare parameters
print("FULL FINETUNING:")
full_stats = count_parameters(model_full)

print("\nLoRA FINETUNING (rank=8):")
lora_stats = count_parameters(model_lora)

In [ ]:
# Visual comparison

methods = ['Full Finetune', 'LoRA (r=4)', 'LoRA (r=8)', 'LoRA (r=16)', 'Frozen + Head']
total_p = [full_stats['total']] * 4 + [full_stats['total']]

dim = 256
n_linear = len(lora_layers)
trainable_p = [
    full_stats['total'],
    n_linear * 2 * dim * 4,
    n_linear * 2 * dim * 8,
    n_linear * 2 * dim * 16,
    256 * 10,  # just the head
]

fig = plot_parameter_comparison(total_p, trainable_p, methods)
plt.savefig('../assets/lora_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Rank Selection Guide

| Rank | Use Case | Memory | Quality |
|------|----------|--------|--------|
| **r=2-4** | Simple tasks, small dataset | Minimal | Good for classification |
| **r=8** | General purpose (recommended start) | Low | Good balance |
| **r=16** | Complex tasks, domain adaptation | Medium | Better |
| **r=32-64** | Significant distribution shift | Higher | Best (diminishing returns) |

**Rule of thumb:** Start with r=8, increase if underfitting, decrease if overfitting.

---
**Next:** `02_qlora_4bit_finetuning.ipynb` - Even more memory savings with quantization!